# Defensive Cybersecurity 50M: expanded defensive training run

This notebook trains a 49M-parameter from-scratch decoder-only Transformer on a Kaggle GPU. It fetches licensed sources at runtime, records filtered source counts, performs general-text pretraining followed by answer-masked defensive SFT, and evaluates on the separate expert-reviewed QA holdout. This is a research checkpoint, not a certified cybersecurity expert.


In [ ]:
import glob, os, shutil, subprocess, sys, torch, tokenizers
if not torch.cuda.is_available(): raise RuntimeError('Kaggle did not provision a GPU; select GPU T4 and retry.')
try:
    import datasets
except ImportError:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'datasets'], check=True)
    import datasets
print('GPU:', torch.cuda.get_device_name(0))
print('PyTorch:', torch.__version__, '| Tokenizers:', tokenizers.__version__, '| Datasets:', datasets.__version__)


In [ ]:
# Copy project utilities and the private base corpus from the attached private Kaggle input.
input_root = '/kaggle/input'
print('Kaggle inputs:', os.listdir(input_root))
train_scripts = glob.glob(os.path.join(input_root, '**', 'train_gpt.py'), recursive=True)
if not train_scripts: raise FileNotFoundError('No attached private input dataset with train_gpt.py found')
bundle = os.path.dirname(train_scripts[0])
work = '/kaggle/working/cyber_50m'
data_dir = '/kaggle/working/data'
os.makedirs(work, exist_ok=True); os.makedirs(data_dir, exist_ok=True)
for name in ['train_gpt.py', 'train_sft.py', 'fetch_additional_sources.py', 'evaluate_tasks.py', 'model_config.json']:
    shutil.copy2(os.path.join(bundle, name), os.path.join(work, name))
for name in ['official_defensive_text.jsonl', 'incident_text.jsonl', 'cybersecurity_qa_train.jsonl', 'holycyber_defensive_train.jsonl', 'cybersecurity_qa_eval.jsonl', 'copilot_style_private.jsonl']:
    source = os.path.join(bundle, name)
    if os.path.exists(source): shutil.copy2(source, os.path.join(data_dir, name))
if not os.path.exists(os.path.join(data_dir, 'cybersecurity_qa_eval.jsonl')): raise FileNotFoundError('Separate held-out QA evaluation file is missing')
subprocess.run([sys.executable, os.path.join(work, 'fetch_additional_sources.py')], check=True)
print('Prepared source files:', sorted(os.listdir(data_dir)))
print('Additional source audit:', open(os.path.join(data_dir, 'additional_sources_summary.json'), encoding='utf-8').read())


In [ ]:
# Train the exact configured ~49.4M parameter model from scratch, then adapt it with answer-only SFT.
pretrain_dir = '/kaggle/working/cyber_50m_pretrain'
subprocess.run([sys.executable, os.path.join(work, 'train_gpt.py'), '--data-dir', data_dir, '--config', os.path.join(work, 'model_config.json'), '--steps', '16000', '--batch-size', '2', '--grad-accum', '4', '--eval-interval', '200', '--patience-evals', '10', '--out-dir', pretrain_dir], check=True)
subprocess.run([sys.executable, os.path.join(work, 'evaluate_tasks.py'), '--checkpoint', os.path.join(pretrain_dir, 'best_checkpoint.pt'), '--tokenizer', os.path.join(pretrain_dir, 'tokenizer.json'), '--eval-file', os.path.join(data_dir, 'cybersecurity_qa_eval.jsonl'), '--output-dir', pretrain_dir, '--max-new-tokens', '96'], check=True)
sft_dir = '/kaggle/working/cyber_50m_sft'
subprocess.run([sys.executable, os.path.join(work, 'train_sft.py'), '--data-dir', data_dir, '--checkpoint', os.path.join(pretrain_dir, 'best_checkpoint.pt'), '--tokenizer', os.path.join(pretrain_dir, 'tokenizer.json'), '--steps', '4000', '--batch-size', '2', '--grad-accum', '4', '--learning-rate', '5e-5', '--eval-interval', '200', '--patience-evals', '10', '--out-dir', sft_dir], check=True)
subprocess.run([sys.executable, os.path.join(work, 'evaluate_tasks.py'), '--checkpoint', os.path.join(sft_dir, 'best_sft_checkpoint.pt'), '--tokenizer', os.path.join(pretrain_dir, 'tokenizer.json'), '--eval-file', os.path.join(data_dir, 'cybersecurity_qa_eval.jsonl'), '--output-dir', sft_dir, '--max-new-tokens', '128'], check=True)


In [ ]:
import json, os
for label, path in [('additional_source_audit', '/kaggle/working/data/additional_sources_summary.json'), ('pretrain_summary', '/kaggle/working/cyber_50m_pretrain/run_summary.json'), ('pretrain_task_eval', '/kaggle/working/cyber_50m_pretrain/task_evaluation_summary.json'), ('sft_summary', '/kaggle/working/cyber_50m_sft/sft_summary.json'), ('sft_task_eval', '/kaggle/working/cyber_50m_sft/task_evaluation_summary.json')]:
    print('\n' + label + ':\n' + open(path, encoding='utf-8').read())
print('SFT artifacts:', os.listdir('/kaggle/working/cyber_50m_sft'))
